# LoRA fine-tuning of BERT on a GLUE task: Kaggle feasibility test

**Goal:** find out whether we can fine-tune a BERT model with LoRA on Kaggle's free GPU, how long it takes, and how much GPU memory it needs.

**Big picture (what happens in this notebook):**
1. **Setup**: install `peft`, import libraries, check the GPU
2. **Config**: all settings in ONE cell (task, model, LoRA, training)
3. **Data**: load a GLUE task from Hugging Face
4. **Tokenize**: turn text into numbers BERT understands
5. **Model + LoRA**: load BERT, freeze it, attach small trainable LoRA adapters
6. **Train**: fine-tune with the Hugging Face `Trainer`, measuring time and memory
7. **Evaluate + feasibility report**: accuracy, time, memory, and an estimate for a full run
8. **Save**: store the small LoRA adapter

**Before you run:** in the Kaggle right sidebar set **Accelerator = GPU (T4 or P100)** and **Internet = On** (needed to download the model, the dataset and `peft`).

## 1. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # use ONE gpu, so time/memory numbers are clean
os.environ["WANDB_DISABLED"] = "true"         # no Weights&Biases login prompt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# PROBLEM WE FACED:
#   Kaggle comes with an old "torchao" (0.10.0) preinstalled. Installing peft can pull in a newer
#   "transformers", which refuses to import with: "ImportError: Found an incompatible version of
#   torchao. Found version 0.10.0, but only versions above 0.16.0 are supported".
# WHY THIS FIX:
#   Our notebook does not use torchao (it is only needed for model quantization), so we simply
#   remove it. That makes transformers skip the version check.
# NOTE:
#   If the error still shows up, the old version is still loaded in memory. Restart the session
#   (Run -> Restart session) and run this cell first, before any other import.
!pip uninstall -y torchao
!pip install -q peft

# FALLBACK (only if removing torchao causes a different error): upgrade it instead of removing it
# !pip install -q -U "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import time, json, math, inspect
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import transformers, peft

print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("WARNING: no GPU found. Turn on a GPU accelerator in the Kaggle sidebar.")

torch: 2.11.0+cu128 | transformers: 5.16.1 | peft: 0.20.0
GPU: Tesla T4 | memory: 15.6 GB


## 2. Config: the only cell you need to edit

- **`TASK`**: which GLUE task. Sizes of the training sets (approx.): `rte` 2.5k, `mrpc` 3.7k, `cola` 8.5k, `sst2` 67k, `qnli` 105k, `qqp` 364k, `mnli` 393k.
- **`MAX_TRAIN_SAMPLES`**: use a small number (e.g. 2000) for a quick feasibility test; set `None` for the full training set.
- **`USE_LORA`**: `True` = LoRA, `False` = normal full fine-tuning (handy to compare time and memory).
- **LoRA settings**: `LORA_R` is the adapter "rank" (bigger = more trainable parameters), `LORA_ALPHA` is a scaling factor (commonly 2x the rank).

In [3]:
# ---------- Task and model ----------
TASK = "sst2"                       # cola, sst2, mrpc, qqp, qnli, rte, mnli
MODEL_NAME = "bert-base-uncased"    # e.g. "distilbert-base-uncased" for a smaller model

# ---------- Data size (for the quick test) ----------
MAX_TRAIN_SAMPLES = None            # None = full training set
MAX_EVAL_SAMPLES = None             # None = full validation set
MAX_LENGTH = 128                    # longer texts are cut to this many tokens

# ---------- LoRA ----------
USE_LORA = True
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["query", "value"]   # the attention layers of BERT that get adapters

# ---------- Training ----------
EPOCHS = 3                          # use 3 for a real run
BATCH_SIZE = 32
LEARNING_RATE = 2e-4 if USE_LORA else 2e-5   # LoRA needs a higher learning rate than full fine-tuning
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
USE_FP16 = HAS_GPU                  # mixed precision: faster and uses less memory on GPU
SEED = 42
OUTPUT_DIR = "/kaggle/working/outputs"

## 3. Load the GLUE data

GLUE tasks differ in their columns: some have one sentence, some have two (a pair). The table below tells the tokenizer which columns to use.

In [4]:
# task -> (first text column, second text column or None)
TASK_COLUMNS = {
    "cola": ("sentence", None),
    "sst2": ("sentence", None),
    "mrpc": ("sentence1", "sentence2"),
    "qqp":  ("question1", "question2"),
    "qnli": ("question", "sentence"),
    "rte":  ("sentence1", "sentence2"),
    "mnli": ("premise", "hypothesis"),
}
assert TASK in TASK_COLUMNS, f"TASK must be one of {list(TASK_COLUMNS)}"
TEXT_A, TEXT_B = TASK_COLUMNS[TASK]
NUM_LABELS = 3 if TASK == "mnli" else 2
EVAL_SPLIT = "validation_matched" if TASK == "mnli" else "validation"

raw = load_dataset("nyu-mll/glue", TASK)
print(raw)

train_raw = raw["train"]
eval_raw = raw[EVAL_SPLIT]
FULL_TRAIN_SIZE = len(train_raw)          # remember it for the time estimate later

if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
if MAX_EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))

print("\nExample:", train_raw[0])
print(f"Using {len(train_raw)} training and {len(eval_raw)} evaluation examples (full train set: {FULL_TRAIN_SIZE})")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Example: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
Using 67349 training and 872 evaluation examples (full train set: 67349)


## 4. Tokenize

BERT cannot read text, only numbers. The tokenizer splits text into tokens and maps each to an ID. We do **not** pad everything to the same length here. The data collator pads each batch only as much as needed, which saves time.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    if TEXT_B is None:
        return tokenizer(batch[TEXT_A], truncation=True, max_length=MAX_LENGTH)
    return tokenizer(batch[TEXT_A], batch[TEXT_B], truncation=True, max_length=MAX_LENGTH)

def prepare(ds):
    ds = ds.map(tokenize, batched=True)
    drop = [c for c in ds.column_names if c not in ("input_ids", "attention_mask", "token_type_ids", "label")]
    return ds.remove_columns(drop)

train_ds = prepare(train_raw)
eval_ds = prepare(eval_raw)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print("Decoded example:", tokenizer.decode(train_ds[0]["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 67349
})
Decoded example: [CLS] hide new secretions from the parental units [SEP]


## 5. Model + LoRA

**What LoRA does:** instead of updating all ~110 million BERT weights, we **freeze** BERT and add tiny trainable matrices ("adapters") next to the attention layers. Only those adapters (plus the new classification layer on top) are trained. That is why LoRA needs less memory and produces a very small file.

In [6]:
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

if USE_LORA:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,        # sequence classification
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

TOTAL_PARAMS, TRAINABLE_PARAMS = count_params(model)
print(f"Total parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,} ({100 * TRAINABLE_PARAMS / TOTAL_PARAMS:.2f}%)")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     109,780,228
Trainable parameters: 296,450 (0.27%)


## 6. Training setup

We use the Hugging Face `Trainer`, which runs the training loop for us (batching, GPU, mixed precision, evaluation). `compute_metrics` turns the model's outputs into the usual GLUE scores.

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = {"accuracy": accuracy_score(labels, preds)}
    if TASK in ("mrpc", "qqp"):
        out["f1"] = f1_score(labels, preds)
    if TASK == "cola":
        out["matthews"] = matthews_corrcoef(labels, preds)
    return out

steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS

# the argument was renamed in newer transformers versions; pick whichever exists
_ta = inspect.signature(TrainingArguments.__init__).parameters
eval_kw = {"eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy": "epoch"}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=int(WARMUP_FRACTION * total_steps),
    fp16=USE_FP16,
    logging_steps=50,
    save_strategy="no",              # don't save checkpoints (saves disk and time); we save the adapter at the end
    report_to="none",
    seed=SEED,
    label_names=["labels"],
    remove_unused_columns=False,     # we already removed the unneeded columns ourselves
    dataloader_num_workers=2,
    **eval_kw,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"{steps_per_epoch} steps per epoch, {total_steps} steps in total")

2105 steps per epoch, 6315 steps in total


## Testing Before Training

In [8]:
# Accuracy BEFORE training.
# Why: the classifier head is randomly initialized, so we expect roughly chance-level
# accuracy (~50% for 2 classes, ~33% for 3). If it is already high, something is wrong.
# If it is low before and high after training, we know the training really did something.

# We evaluate on a fixed subset of the training data (full train sets can be huge and slow to evaluate)
N_TRAIN_CHECK = min(2000, len(train_ds))
train_check_ds = train_ds.select(range(N_TRAIN_CHECK))

def evaluate_both(tag):
    train_m = trainer.evaluate(eval_dataset=train_check_ds, metric_key_prefix="train")
    val_m = trainer.evaluate(eval_dataset=eval_ds, metric_key_prefix="val")
    return {
        f"{tag}_train_acc": train_m["train_accuracy"], f"{tag}_train_loss": train_m["train_loss"],
        f"{tag}_val_acc": val_m["val_accuracy"], f"{tag}_val_loss": val_m["val_loss"],
    }

before = evaluate_both("before")
print("BEFORE training:", {k: round(v, 4) for k, v in before.items()})

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.687433,0,0.557000


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.691378,0,0.510321


BEFORE training: {'before_train_acc': 0.557, 'before_train_loss': 0.6874, 'before_val_acc': 0.5103, 'before_val_loss': 0.6914}


## 7. Train (and measure time + GPU memory)

In [9]:
if HAS_GPU:
    torch.cuda.reset_peak_memory_stats()

start = time.time()
train_result = trainer.train()
TRAIN_SECONDS = time.time() - start

PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1e9 if HAS_GPU else 0.0
print(f"\nTraining finished in {TRAIN_SECONDS:.0f} s ({TRAIN_SECONDS / 60:.1f} min)")
print(f"Peak GPU memory: {PEAK_GPU_GB:.2f} GB")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.244543,0.242193,0.905963
2,0.188130,0.238261,0.907110
3,0.188500,0.241080,0.903670



Training finished in 359 s (6.0 min)
Peak GPU memory: 1.32 GB


## 8. Evaluate + feasibility report

In [10]:
metrics = trainer.evaluate()
print("Validation metrics:", {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

samples_processed = len(train_ds) * EPOCHS
sec_per_sample = TRAIN_SECONDS / samples_processed
full_run_hours = sec_per_sample * FULL_TRAIN_SIZE * 3 / 3600     # estimate: full train set, 3 epochs

print("\n" + "=" * 55)
print("FEASIBILITY REPORT")
print("=" * 55)
print(f"Task / model:              {TASK} / {MODEL_NAME}")
print(f"Method:                    {'LoRA (r=%d)' % LORA_R if USE_LORA else 'full fine-tuning'}")
print(f"Trainable parameters:      {TRAINABLE_PARAMS:,} of {TOTAL_PARAMS:,} ({100 * TRAINABLE_PARAMS / TOTAL_PARAMS:.2f}%)")
print(f"Training examples used:    {len(train_ds)} x {EPOCHS} epoch(s)")
print(f"Training time:             {TRAIN_SECONDS / 60:.1f} min")
print(f"Throughput:                {samples_processed / TRAIN_SECONDS:.1f} examples/sec")
print(f"Peak GPU memory:           {PEAK_GPU_GB:.2f} GB")
print(f"ESTIMATE full run:         ~{full_run_hours:.2f} h for {FULL_TRAIN_SIZE} examples x 3 epochs")
print("=" * 55)
print("(The estimate extrapolates from this run; real time varies a bit. Kaggle sessions last ~12 h max and the weekly GPU quota is limited, so check your quota in Kaggle.)")

report = {
    "task": TASK, "model": MODEL_NAME, "use_lora": USE_LORA, "lora_r": LORA_R,
    "total_params": TOTAL_PARAMS, "trainable_params": TRAINABLE_PARAMS,
    "train_examples_used": len(train_ds), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "train_seconds": TRAIN_SECONDS, "examples_per_sec": samples_processed / TRAIN_SECONDS,
    "peak_gpu_gb": PEAK_GPU_GB, "estimated_full_run_hours_3_epochs": full_run_hours,
    "metrics": {k: v for k, v in metrics.items() if isinstance(v, float)},
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "feasibility_report.json"), "w") as f:
    json.dump(report, f, indent=2)

Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.241080,3,0.903670


Validation metrics: {'eval_loss': 0.2411, 'eval_accuracy': 0.9037}

FEASIBILITY REPORT
Task / model:              sst2 / bert-base-uncased
Method:                    LoRA (r=8)
Trainable parameters:      296,450 of 109,780,228 (0.27%)
Training examples used:    67349 x 3 epoch(s)
Training time:             6.0 min
Throughput:                562.1 examples/sec
Peak GPU memory:           1.32 GB
ESTIMATE full run:         ~0.10 h for 67349 examples x 3 epochs
(The estimate extrapolates from this run; real time varies a bit. Kaggle sessions last ~12 h max and the weekly GPU quota is limited, so check your quota in Kaggle.)


## 9. Save the adapter

With LoRA, only the small adapter (a few MB) is saved, not the whole BERT model. To use it later, load the base model again and attach the adapter with `PeftModel.from_pretrained`.

In [11]:
SAVE_DIR = os.path.join(OUTPUT_DIR, "lora_adapter" if USE_LORA else "full_model")
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, f)) for f in os.listdir(SAVE_DIR)) / 1e6
print(f"Saved to {SAVE_DIR} ({size_mb:.1f} MB)")

Saved to /kaggle/working/outputs/lora_adapter (1.9 MB)


In [12]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): L

## Testing After Training

In [13]:
# Accuracy AFTER training, compared with BEFORE
after = evaluate_both("after")

print(f"\n{'':<22}{'before':>10}{'after':>10}")
print("-" * 42)
for name, key in [("Train accuracy", "train_acc"), ("Train loss", "train_loss"),
                  ("Validation accuracy", "val_acc"), ("Validation loss", "val_loss")]:
    print(f"{name:<22}{before['before_' + key]:>10.4f}{after['after_' + key]:>10.4f}")

# Quick interpretation
if after["after_val_acc"] > before["before_val_acc"] + 0.1:
    print("\nOK: validation accuracy clearly improved, so the training worked.")
else:
    print("\nWARNING: little or no improvement. Check the learning rate, labels, or number of epochs.")

gap = after["after_train_acc"] - after["after_val_acc"]
if gap > 0.1:
    print(f"Note: train accuracy is {gap:.1%} higher than validation, which suggests overfitting.")

Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.146073,3,0.946500


Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.241080,3,0.903670



                          before     after
------------------------------------------
Train accuracy            0.5570    0.9465
Train loss                0.6874    0.1461
Validation accuracy       0.5103    0.9037
Validation loss           0.6914    0.2411

OK: validation accuracy clearly improved, so the training worked.


## What to do next

1. Run once as-is (2000 examples, 1 epoch) to see if everything works and how fast it is.
2. Set `MAX_TRAIN_SAMPLES = None` and `EPOCHS = 3` for a real run (or compare against `USE_LORA = False`).
3. Copy the numbers from the feasibility report into your notes. They are useful for the paper.

---
# NEW Study: LoRA settings on more datasets

Everything below is new, the cells above are unchanged. The new part reuses the config, tokenizer, data collator and `Trainer` setup from above.

**What is compared, on each dataset (SST-2, AG News, 20 Newsgroups):**

| run | what is trained | how |
|---|---|---|
| **Baseline** | only a logistic regression on top | frozen BERT, mean vector of the last layer, scikit-learn, no training steps |
| **Reference** | LoRA matrices A, B + new classifier layer | exactly the config cell above (`LORA_R`, `LORA_TARGET_MODULES`, `LEARNING_RATE`, `MAX_LENGTH`, all 12 layers) |
| **Variations** | LoRA matrices A, B + new classifier layer | the reference with **one** setting changed, see the table below |

BERT's own weights are frozen in every run. Learning rate, epochs, warmup, batch size and weight decay are the training settings of the trainable parts only (LoRA A, B and the classifier layer).

**Variations and why** (details and page numbers in `study_settings.md`):

| group | values | paper that motivates it |
|---|---|---|
| layers with LoRA | all 12 (reference), 1-3, 4-6, 7-9, 10-12 | Zhang et al. 2023 (AdaLoRA) Fig. 1b, Merchant et al. 2020 Sec. 5 |
| modules | query + value (reference), all attention, feed-forward, all linear | AdaLoRA Fig. 1a and Tab. 14, Hu et al. 2022 Tab. 5 |
| rank | 1, `LORA_R` (reference), 64 | Hu et al. 2022 Tab. 6, AdaLoRA Tab. 5 |
| learning rate | `LEARNING_RATE` / 4, `LEARNING_RATE` (reference), `LEARNING_RATE` x 5 | He et al. 2026 Fig. 6, Hu et al. 2022 Tab. 9 |
| text length, 20 Newsgroups only | first 128 tokens (reference), first 512, last 128 | Sun et al. 2019 Tab. 2 |

Layer and module variations keep about the same number of trainable weights as the reference (higher or lower rank), so they compare *where* LoRA sits, not *how much* of it.

**To skip the SST-2 run above:** run the cells up to "6. Training setup", then continue here. All results are on a dev split, the test split stays untouched for the paper.

In [ ]:
# NEW Study: settings (the only new cell you need to edit)
import glob, re, shutil
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import train_test_split

if "trainer" in globals():  # free the GPU from the feasibility run above
    del trainer, model
    torch.cuda.empty_cache()

# ---------- 1. Size of the study ----------
STUDY_QUICK = True                  # True = short test of all new cells (30 steps), False = the real run
STUDY_DATASETS = ["sst2", "ag_news", "20newsgroups"]
STUDY_SEEDS = [0, 1] if STUDY_QUICK else [0, 1, 2]
STUDY_TRAIN_SAMPLES = 500 if STUDY_QUICK else 20000
STUDY_DEV_SAMPLES = 200 if STUDY_QUICK else 5000      # sst2 only has 436 dev sentences
STUDY_PROBE_SAMPLES = 200 if STUDY_QUICK else 2000    # kept apart for the XAI part
STUDY_DIR = os.path.join(OUTPUT_DIR, "study")

# ---------- 2. Training settings of the trainable parts (LoRA A, B + classifier layer) ----------
# Same for every run. BERT itself is frozen and gets no updates.
# Taken from the config cell above: BATCH_SIZE, WEIGHT_DECAY, WARMUP_FRACTION, USE_FP16
STUDY_EPOCHS = {"sst2": 2, "ag_news": 2, "20newsgroups": 4}   # 20newsgroups has fewer texts, so more epochs
STUDY_QUICK_STEPS = 30                                         # used instead of epochs when STUDY_QUICK

# ---------- 3. Reference run = the LoRA config cell above ----------
REFERENCE = dict(r=LORA_R, modules=LORA_TARGET_MODULES, layers=None,  # layers=None: all 12 blocks
                 lr=LEARNING_RATE, max_len=MAX_LENGTH, cut="right")    # cut="right" keeps the start of a text

# ---------- 4. >>> LoRA VARIATIONS: the reference with one setting changed <<< ----------
#   r       = LoRA rank                      -> LoraConfig(r=...)
#   modules = which matrices get LoRA        -> LoraConfig(target_modules=...)
#   layers  = which of the 12 blocks (0-11)  -> LoraConfig(layers_to_transform=...)
#   lr      = learning rate                  -> TrainingArguments(learning_rate=...)
#   max_len, cut = how the text is cut       -> tokenizer(max_length=...), truncation side
ATTENTION = ["query", "key", "value", "attention.output.dense"]
FEED_FORWARD = r".*\.layer\.\d+\.(intermediate|output)\.dense"     # the two feed-forward layers of each block
ALL_LINEAR = r".*\.layer\.\d+\..*(query|key|value|dense)"          # every linear layer inside the 12 blocks

def change(**kw):
    return {**REFERENCE, **kw}

STUDY_SETTINGS = {
    "reference":        ("reference", REFERENCE),
    # where in the depth: 3 blocks each, rank x4 so the number of weights stays the same (AdaLoRA Fig. 1b, Merchant 2020)
    "layers 1-3":       ("layers", change(layers=[0, 1, 2], r=4 * LORA_R)),
    "layers 4-6":       ("layers", change(layers=[3, 4, 5], r=4 * LORA_R)),
    "layers 7-9":       ("layers", change(layers=[6, 7, 8], r=4 * LORA_R)),
    "layers 10-12":     ("layers", change(layers=[9, 10, 11], r=4 * LORA_R)),
    # which matrices, rank lowered so the number of weights stays about the same (AdaLoRA Fig. 1a, Tab. 14, Hu 2022 Tab. 5)
    "all attention":    ("modules", change(modules=ATTENTION, r=LORA_R // 2)),
    "feed-forward":     ("modules", change(modules=FEED_FORWARD, r=round(0.4 * LORA_R))),
    "all linear":       ("modules", change(modules=ALL_LINEAR, r=max(1, LORA_R // 4))),
    # how many directions the update can have (Hu 2022 Tab. 6, AdaLoRA Tab. 5)
    "rank 1":           ("rank", change(r=1)),
    "rank 64":          ("rank", change(r=64)),
    # step size (He 2026 Fig. 6, Hu 2022 Tab. 9)
    "lr low":           ("lr", change(lr=LEARNING_RATE / 4)),
    "lr high":          ("lr", change(lr=LEARNING_RATE * 5)),
    # which part of a long text BERT sees, only 20newsgroups has texts over 128 tokens (Sun 2019 Tab. 2)
    "first 512 tokens": ("length", change(max_len=512)),
    "last 128 tokens":  ("length", change(cut="left")),
}
STUDY_GROUPS = ["layers", "modules", "rank", "lr", "length"]
PUBLISHED = {"sst2": 0.929, "ag_news": 0.9475, "20newsgroups": 0.861}   # full fine-tuning: Devlin 2019, Sun 2019, Dai 2022

def what_changed(setting):
    """Short text: what differs from the reference, e.g. 'layers=[9, 10, 11], r=32'."""
    s = STUDY_SETTINGS[setting][1]
    diff = [f"{k}={v}" for k, v in s.items() if v != REFERENCE[k]]
    return ", ".join(diff) if diff else "nothing (reference)"

def runs_for(dataset):
    return [n for n, (g, _) in STUDY_SETTINGS.items() if g != "length" or dataset == "20newsgroups"]

def members(group):
    return [n for n, (g, _) in STUDY_SETTINGS.items() if g == group]

# continue an earlier run: on Kaggle add the output of an earlier version as input
earlier = glob.glob("/kaggle/input/**/study/results.csv", recursive=True)
if earlier and not os.path.exists(os.path.join(STUDY_DIR, "results.csv")):
    shutil.copytree(os.path.dirname(earlier[0]), STUDY_DIR, dirs_exist_ok=True)
os.makedirs(STUDY_DIR, exist_ok=True)

print(f"reference: r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}, all 12 layers, lr={LEARNING_RATE}, max length {MAX_LENGTH}")
for name in STUDY_SETTINGS:
    print(f"  {name:18} {what_changed(name)}")
print(sum(len(runs_for(d)) for d in STUDY_DATASETS) * len(STUDY_SEEDS), "runs in total")

## NEW Study 1: Data

Dev is used for everything here. Test stays untouched for the paper.
- **SST-2**: GLUE hides the test labels, so the validation sentences are split into dev and test (the run above used all of them). Train consists of short phrases, so dev does not come from there.
- **AG News, 20 Newsgroups**: dev comes from train. 20 Newsgroups is the version without mail headers. Empty posts and posts that are in train and test are dropped.

In [ ]:
# NEW Study: load the three datasets
# Task in all three: one whole text -> one class. Accuracy and macro F1 compare the predicted class with the label.
#   sst2          sentiment of a movie review sentence: negative, positive
#   ag_news       topic of a news text (title + description): World, Sports, Business, Sci/Tech
#   20newsgroups  topic of a forum post (real documents): 20 groups, e.g. comp.graphics, rec.autos, sci.space
def load_study_data(name):
    if name == "sst2":
        ds = load_dataset("nyu-mll/glue", "sst2")
        train, val = (ds[s].to_pandas().rename(columns={"sentence": "text"}) for s in ("train", "validation"))
        dev, test = train_test_split(val, test_size=0.5, stratify=val.label, random_state=0)
        names = ["negative", "positive"]
    else:
        ds = load_dataset("fancyzhx/ag_news" if name == "ag_news" else "SetFit/20_newsgroups")
        train, test = ds["train"].to_pandas(), ds["test"].to_pandas()
        if name == "ag_news":
            names = ds["train"].features["label"].names
        else:
            names = [n for _, n in sorted(set(zip(train.label, train.label_text)))]
        test = test[test.text.str.strip() != ""]
        train = train[(train.text.str.strip() != "") & ~train.text.isin(test.text)].drop_duplicates("text")
        train, dev = train_test_split(train, test_size=min(STUDY_DEV_SAMPLES, len(train) // 6), stratify=train.label, random_state=0)
    train = train[~train.text.isin(dev.text)][["text", "label"]].sample(frac=1, random_state=0).reset_index(drop=True)
    dev = dev[["text", "label"]].sample(frac=1, random_state=0).head(STUDY_DEV_SAMPLES).reset_index(drop=True)
    return {"train": train[:-STUDY_PROBE_SAMPLES][:STUDY_TRAIN_SAMPLES].reset_index(drop=True),
            "probe": train[-STUDY_PROBE_SAMPLES:].reset_index(drop=True), "dev": dev,
            "test_size": len(test), "num_labels": train.label.nunique(), "class_names": names}

STUDY_DATA = {name: load_study_data(name) for name in STUDY_DATASETS}

def tokenized(dataset, split, s):
    """Hugging Face dataset with token ids, cut at max_len from the right (keeps the start) or the left (keeps the end)."""
    tokenizer.truncation_side = s["cut"]
    ds = Dataset.from_pandas(STUDY_DATA[dataset][split], preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=s["max_len"]), batched=True, remove_columns=["text"])

rows = []
for name, d in STUDY_DATA.items():
    tokens = np.array([len(t) for t in tokenizer(d["dev"].text.tolist()).input_ids])
    rows.append({"dataset": name, "classes": d["num_labels"],
                 "class_names": ", ".join(d["class_names"][:4]) + (", ..." if len(d["class_names"]) > 4 else ""), "train": len(d["train"]), "dev": len(d["dev"]),
                 "test_untouched": d["test_size"], "median_tokens": int(np.median(tokens)),
                 "above_128": f"{np.mean(tokens > 128):.0%}", "above_512": f"{np.mean(tokens > 512):.0%}"})
data_table = pd.DataFrame(rows)
data_table

### NEW Study 1b: Which classes?

AG News and 20 Newsgroups are **topic** classification, SST-2 is **sentiment**. One row per class: name, number of texts in train and dev, and one dev example.

In [ ]:
# NEW Study: the classes of each dataset, with counts and one example each
rows = []
for name, d in STUDY_DATA.items():
    for i, cls in enumerate(d["class_names"]):
        examples = d["dev"][d["dev"].label == i].text
        rows.append({"dataset": name, "label": i, "class": cls,
                     "train_texts": int((d["train"].label == i).sum()), "dev_texts": int((d["dev"].label == i).sum()),
                     "example": examples.iloc[0][:120].replace("\n", " ") if len(examples) else ""})
class_table = pd.DataFrame(rows)

# NEW plotting: texts per class in the full dataset (all splits, before we take our subsets)
FULL = {"sst2": ("nyu-mll/glue", "sst2", ["train", "validation"]), "ag_news": ("fancyzhx/ag_news", None, ["train", "test"]),
        "20newsgroups": ("SetFit/20_newsgroups", None, ["train", "test"])}
fig, axes = plt.subplots(1, len(STUDY_DATASETS), figsize=(5 * len(STUDY_DATASETS), 0.3 * max(len(STUDY_DATA[d]["class_names"]) for d in STUDY_DATASETS) + 2), squeeze=False)
for ax, name in zip(axes[0], STUDY_DATASETS):
    path, config, splits = FULL[name]
    ds = load_dataset(path, config)
    names = STUDY_DATA[name]["class_names"]
    y = np.arange(len(names))
    for k, split in enumerate(splits):
        counts = np.bincount(ds[split]["label"], minlength=len(names))
        ax.barh(y + (0.2 if k == 0 else -0.2), counts, height=0.4, label=f"{split} ({counts.sum():,})")
    ax.set(yticks=y, title=name, xlabel="number of texts")
    ax.set_yticklabels(names, fontsize=7)
    ax.legend(fontsize=7)
fig.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
for ext in ("png", "svg"):
    fig.savefig(os.path.join(STUDY_DIR, f"texts_per_class.{ext}"), dpi=200, bbox_inches="tight")
plt.show()

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
class_table

## NEW Study 2: One run = one setting, one dataset, one seed

What happens in one run, step by step:
1. **Data**: tokenize train and dev with the text length of this setting
2. **Model**: BERT (frozen) + a new classifier layer + LoRA matrices A, B where this setting says
3. **Training settings** for the trainable parts: learning rate of this setting, the rest is the same for all runs
4. **Train** with the `Trainer` from above
5. **Accuracy test** on the dev texts: predicted class vs. true class
6. **Save** the small adapter (for XAI) and the prediction of every dev text (for the per-class plots)

In [ ]:
# NEW Study: build, train and score one run
from peft import PeftModel

def study_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

def build_model(s, num_labels, seed):
    """Frozen BERT + new classifier layer + LoRA as given in setting s."""
    set_seed(seed)
    # CLASSIFIER: BERT + a new linear layer on top ([CLS] -> BERT's pooler -> linear layer, 768 -> num_labels).
    # The linear layer starts random and is trained, BERT's own weights stay frozen.
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    # >>> LoRA VARIATIONS go in here (r, modules, layers). Alpha keeps the ratio LORA_ALPHA / LORA_R of the config.
    lora = LoraConfig(task_type=TaskType.SEQ_CLS,      # SEQ_CLS keeps the new classifier layer trainable next to LoRA
                      r=s["r"], lora_alpha=s["r"] * LORA_ALPHA // LORA_R, lora_dropout=LORA_DROPOUT,
                      target_modules=s["modules"], layers_to_transform=s["layers"])
    return get_peft_model(m, lora)

def adapter_dir(dataset, setting, seed):
    return os.path.join(STUDY_DIR, "adapters", dataset, f"{setting.replace(' ', '_')}_seed{seed}")

def run_study(dataset, setting, seed):
    group, s = STUDY_SETTINGS[setting]
    d = STUDY_DATA[dataset]

    # 1. data, cut to the text length of this setting
    train_ds, dev_ds = tokenized(dataset, "train", s), tokenized(dataset, "dev", s)

    # 2. model: frozen BERT + classifier layer + LoRA of this setting
    model = build_model(s, d["num_labels"], seed)

    # 3. training settings of the trainable parts (LoRA A, B + classifier layer)
    total = STUDY_QUICK_STEPS if STUDY_QUICK else math.ceil(len(train_ds) / BATCH_SIZE) * STUDY_EPOCHS[dataset]
    curve = setting == "reference" and seed == STUDY_SEEDS[0]     # learning curve only for the reference
    eval_key = next(iter(eval_kw))                                  # same name trick as in the training setup above
    args = TrainingArguments(
        output_dir=os.path.join(STUDY_DIR, "tmp"),
        num_train_epochs=STUDY_EPOCHS[dataset],
        max_steps=STUDY_QUICK_STEPS if STUDY_QUICK else -1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=s["lr"],                     # >>> learning rate of this setting
        weight_decay=WEIGHT_DECAY,
        warmup_steps=int(WARMUP_FRACTION * total),
        fp16=USE_FP16,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=seed,
        label_names=["labels"],
        remove_unused_columns=False,
        dataloader_num_workers=2,
        **({eval_key: "steps", "eval_steps": max(1, total // 5)} if curve else {eval_key: "no"}),
    )

    # 4. train
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=data_collator, compute_metrics=study_metrics)
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train()
    seconds = time.time() - start

    # 5. ACCURACY TEST: predicted class vs. true class on the dev texts
    out = trainer.predict(dev_ds, metric_key_prefix="eval")
    m = out.metrics
    pred_rows = [{"dataset": dataset, "setting": setting, "seed": seed, "text": i, "label": int(l), "pred": int(p)}
                 for i, (l, p) in enumerate(zip(out.label_ids, out.predictions.argmax(-1)))]

    # 6. save the adapter, return the scores
    trainer.model.save_pretrained(adapter_dir(dataset, setting, seed))
    curve_rows = [{"dataset": dataset, "step": h["step"], "accuracy": h["eval_accuracy"]} for h in trainer.state.log_history if "eval_accuracy" in h]
    result = {"dataset": dataset, "group": group, "setting": setting, "what_changed": what_changed(setting), "seed": seed,
              "accuracy": m["eval_accuracy"], "macro_f1": m["eval_macro_f1"], "seconds": seconds,
              "trainable": count_params(model)[1], "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if HAS_GPU else 0.0}
    del trainer, model
    if HAS_GPU:
        torch.cuda.empty_cache()
    return result, curve_rows, pred_rows

# check: the reference must have as many trainable weights as the feasibility run above (296,450 on SST-2)
print("trainable weights (LoRA + classifier) on a 2-class dataset:")
for name in STUDY_SETTINGS:
    print(f"  {name:18} {count_params(build_model(STUDY_SETTINGS[name][1], 2, 0))[1]:>10,}")

### NEW Study 2b: How the scores are computed

For $N$ dev texts with true class $y_i$ and $C$ classes. The model gives one score per class, the predicted class is the highest one:
$$\hat y_i = \arg\max_c \; (W v_i + b)_c ,$$
where $v_i$ is the text vector ([CLS] after BERT's pooler) and $W, b$ the linear layer on top. Training minimises the cross entropy $-\log \operatorname{softmax}(W v_i + b)_{y_i}$.

- **Accuracy**: share of texts with the right class, $\;\text{acc} = \frac{1}{N} \sum_i [\hat y_i = y_i]$
- **Accuracy per class** (recall of class $c$): right ones among the texts of class $c$, $\;R_c = \frac{\#\{\hat y_i = y_i = c\}}{\#\{y_i = c\}}$
- **Precision of class $c$**: right ones among the texts predicted as $c$, $\;P_c = \frac{\#\{\hat y_i = y_i = c\}}{\#\{\hat y_i = c\}}$
- **Macro F1**: F1 per class, then the plain mean, so every class counts the same no matter how many texts it has, $\;F1_c = \frac{2 P_c R_c}{P_c + R_c}, \quad \text{macro F1} = \frac{1}{C}\sum_c F1_c$
- **Over seeds**: mean and standard deviation of the 3 runs, lowest to highest seed as bars in the plots
- **Change vs. reference**: $100 \cdot (\text{acc}_\text{setting} - \text{acc}_\text{reference})$, in points, both means over seeds
- **Confusion matrix**: row $c$, column $c'$ = share of texts of class $c$ predicted as $c'$, the diagonal is $R_c$

## NEW Study 3: Baseline, BERT as it is

Frozen BERT, mean over the tokens of the last layer, logistic regression on top. The line the LoRA runs have to beat.

In [ ]:
# NEW Study: frozen BERT + logistic regression
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DEVICE = "cuda" if HAS_GPU else "cpu"
frozen_bert = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False).to(DEVICE).eval()

@torch.no_grad()
def layer_vectors(model, texts):
    """One vector per text and layer (mean over the tokens), shape [13, texts, 768]. Layer 0 = embeddings."""
    tokenizer.truncation_side = "right"
    out, texts = [], list(texts)
    for i in range(0, len(texts), 64):
        b = tokenizer(texts[i:i + 64], truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
            hidden = model(**b, output_hidden_states=True).hidden_states
        mask = b["attention_mask"].unsqueeze(-1).float()
        out.append(torch.stack([(h.float() * mask).sum(1) / mask.sum(1) for h in hidden]).cpu())
    return torch.cat(out, 1).numpy()

baseline = {}
for name, d in STUDY_DATA.items():
    X, X_dev = layer_vectors(frozen_bert, d["train"].text)[-1], layer_vectors(frozen_bert, d["dev"].text)[-1]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    best_c = GridSearchCV(clf, {"logisticregression__C": [0.001, 0.01, 0.1, 1]}, cv=3).fit(X[:5000], d["train"].label[:5000]).best_params_
    preds = clf.set_params(**best_c).fit(X, d["train"].label).predict(X_dev)
    baseline[name] = {"accuracy": accuracy_score(d["dev"].label, preds), "macro_f1": f1_score(d["dev"].label, preds, average="macro")}
baseline = pd.DataFrame(baseline).T
baseline

## NEW Study 4: Run all settings

For every dataset: the reference and each variation, each with every seed. The output shows what changed compared to the reference. If the notebook stops, run this cell again, finished runs are skipped.

In [ ]:
# NEW Study: run all settings
results_file, curves_file = os.path.join(STUDY_DIR, "results.csv"), os.path.join(STUDY_DIR, "curves.csv")
predictions_file = os.path.join(STUDY_DIR, "predictions.csv")   # true and predicted class of every dev text, every run
results = pd.read_csv(results_file).to_dict("records") if os.path.exists(results_file) else []
curves = pd.read_csv(curves_file).to_dict("records") if os.path.exists(curves_file) else []
predictions = pd.read_csv(predictions_file).to_dict("records") if os.path.exists(predictions_file) else []
done = {(r["dataset"], r["setting"], r["seed"]) for r in results}

start = time.time()
for dataset in STUDY_DATASETS:
    print(f"--- {dataset}: frozen baseline {baseline.accuracy[dataset]:.3f}, then reference and variations ---")
    for setting in runs_for(dataset):
        for seed in STUDY_SEEDS:
            if (dataset, setting, seed) in done:
                continue
            result, curve_rows, pred_rows = run_study(dataset, setting, seed)
            results.append(result)
            curves += curve_rows
            predictions += pred_rows
            pd.DataFrame(results).to_csv(results_file, index=False)
            pd.DataFrame(curves).to_csv(curves_file, index=False)
            pd.DataFrame(predictions).to_csv(predictions_file, index=False)
            print(f"{dataset:12} {setting:18} seed {seed}  accuracy {result['accuracy']:.3f}  {result['seconds']:.0f}s   changed: {result['what_changed']}")

study = pd.DataFrame(results)
print(f"done in {(time.time() - start) / 60:.0f} min")
if STUDY_QUICK:   # estimate for the real run: 20000 training texts (about 7000 on 20newsgroups), 3 seeds
    full_steps = study.dataset.map(lambda d: math.ceil((7000 if d == "20newsgroups" else 20000) / BATCH_SIZE) * STUDY_EPOCHS[d])
    hours = (study.seconds / STUDY_QUICK_STEPS * full_steps).sum() * 3 / len(STUDY_SEEDS) / 3600
    print(f"The real run needs about {hours:.1f} h of training on this machine.")

---
# NEW plotting: results of the parameter variation

1. **Learning curve** of the reference: should be flat at the end, otherwise raise `STUDY_EPOCHS`.
2. **Accuracy per setting**: dot = mean over seeds, bar = lowest to highest seed. Grey dashed = frozen baseline, black dotted = reference, red = published full fine-tuning (other splits, only for orientation).
3. **Change per dataset**: accuracy change compared to the reference of the same dataset, in points.
4. **Spread per group**: best minus worst setting of a group. `visible` = more than twice the typical seed difference. A rule of thumb, not a test.

All plots are saved as PNG and SVG in `study/`.

In [ ]:
# NEW plotting: learning curve and accuracy per setting
def save_plot(fig, name):
    plt.rcParams["svg.fonttype"] = "none"   # text stays editable in the SVG
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(os.path.join(STUDY_DIR, f"{name}.{ext}"), dpi=200, bbox_inches="tight")
    plt.show()

cur = pd.DataFrame(curves)
fig, axes = plt.subplots(1, len(STUDY_DATASETS), figsize=(4.5 * len(STUDY_DATASETS), 3), squeeze=False)
for ax, d in zip(axes[0], STUDY_DATASETS):
    c = cur[cur.dataset == d]
    ax.plot(c.step, c.accuracy, marker="o")
    ax.axhline(baseline.accuracy[d], color="grey", ls="--")
    ax.set(title=d, xlabel="training step", ylabel="dev accuracy (reference)")
save_plot(fig, "learning_curve")

names = list(STUDY_SETTINGS)[::-1]
fig, axes = plt.subplots(1, len(STUDY_DATASETS), figsize=(4.8 * len(STUDY_DATASETS), 5), squeeze=False, sharey=True)
for ax, d in zip(axes[0], STUDY_DATASETS):
    r = study[study.dataset == d]
    for y, setting in enumerate(names):
        acc = r[r.setting == setting].accuracy
        if len(acc):
            ax.plot([acc.min(), acc.max()], [y, y], color="C0")
            ax.plot(acc.mean(), y, "o", color="C0")
    ax.axvline(baseline.accuracy[d], color="grey", ls="--")
    ax.axvline(r[r.setting == "reference"].accuracy.mean(), color="black", ls=":")
    ax.axvline(PUBLISHED[d], color="C3")
    ax.set(title=d, xlabel="dev accuracy", yticks=range(len(names)), yticklabels=names)
save_plot(fig, "accuracy_per_setting")

accuracy_table = study.groupby(["dataset", "group", "setting"], sort=False).agg(
    accuracy=("accuracy", "mean"), seed_std=("accuracy", "std"), macro_f1=("macro_f1", "mean"),
    trainable=("trainable", "first"), seconds=("seconds", "mean")).reset_index()
accuracy_table.round(4)

In [ ]:
# NEW plotting: same setting on different datasets
reference_acc = study[study.setting == "reference"].groupby("dataset").accuracy.mean()
study["change"] = (study.accuracy - study.dataset.map(reference_acc)) * 100
change_table = study[study.setting != "reference"].groupby(["group", "setting", "dataset"], sort=False).change.agg(["mean", "min", "max"]).reset_index()

names = [n for n in STUDY_SETTINGS if n != "reference"][::-1]
fig, ax = plt.subplots(figsize=(7, 0.45 * len(names) + 1))
for j, d in enumerate(STUDY_DATASETS):
    e = change_table[change_table.dataset == d].set_index("setting")
    for y, setting in enumerate(names):
        if setting in e.index:
            y2 = y + (j - 1) * 0.22
            ax.plot([e["min"][setting], e["max"][setting]], [y2, y2], color=f"C{j}")
            ax.plot(e["mean"][setting], y2, "o", color=f"C{j}")
    ax.plot([], [], "o", color=f"C{j}", label=d)
ax.axvline(0, color="black", ls=":")
ax.set(yticks=range(len(names)), yticklabels=names, xlabel="accuracy change vs. reference (points)")
ax.legend(fontsize=8)
save_plot(fig, "change_per_dataset")

spread = []
for d, r in study.groupby("dataset", sort=False):
    mean, seed_std = r.groupby("setting").accuracy.mean(), r.groupby("setting").accuracy.std().mean()
    for group in STUDY_GROUPS:
        g = ["reference"] + [n for n in members(group) if n in mean.index]
        if len(g) > 1:
            s = mean[g].max() - mean[g].min()
            spread.append({"dataset": d, "group": group, "best": mean[g].idxmax(), "spread": 100 * s,
                           "seed_std": 100 * seed_std, "visible": s > 2 * seed_std})
spread = pd.DataFrame(spread)
spread.round(2)

### NEW plotting: results per class

Which topics are easy, which get mixed up, and which settings help or hurt which topic. All on dev texts, pooled over seeds.
1. **Accuracy per class** of the reference, sorted: most correct on top.
2. **Confusion matrix** of the reference: row = true class, column = predicted class, share of the row. Off the diagonal = mixed up.
3. **Change per class**: accuracy per class of each setting minus the reference, in points. Green = better, red = worse.

In [ ]:
# NEW plotting: accuracy per class, confusion matrix, change per class
pred_df = pd.DataFrame(predictions)
pred_df["correct"] = pred_df.label == pred_df.pred
per_class = pred_df.groupby(["dataset", "setting", "label"], sort=False).correct.mean().rename("accuracy").reset_index()
per_class["class"] = [STUDY_DATA[d]["class_names"][l] for d, l in zip(per_class.dataset, per_class.label)]

# 1. accuracy per class, reference
fig, axes = plt.subplots(1, len(STUDY_DATASETS), figsize=(4.8 * len(STUDY_DATASETS), 0.3 * max(len(STUDY_DATA[d]["class_names"]) for d in STUDY_DATASETS) + 1.5), squeeze=False)
for ax, d in zip(axes[0], STUDY_DATASETS):
    r = per_class[(per_class.dataset == d) & (per_class.setting == "reference")].sort_values("accuracy")
    ax.barh(r["class"], r.accuracy, color="C0")
    ax.set(title=d, xlabel="dev accuracy (reference)", xlim=(0, 1))
    ax.tick_params(axis="y", labelsize=7)
save_plot(fig, "accuracy_per_class")

# 2. confusion matrix, reference
for d in STUDY_DATASETS:
    names = STUDY_DATA[d]["class_names"]
    r = pred_df[(pred_df.dataset == d) & (pred_df.setting == "reference")]
    cm = pd.crosstab(r.label, r.pred, normalize="index").reindex(index=range(len(names)), columns=range(len(names)), fill_value=0)
    fig, ax = plt.subplots(figsize=(0.35 * len(names) + 2.5, 0.35 * len(names) + 2))
    ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=1)
    ax.set(xticks=range(len(names)), yticks=range(len(names)), xlabel="predicted", ylabel="true", title=f"{d}, reference")
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_yticklabels(names, fontsize=7)
    for i in range(len(names)):
        for j in range(len(names)):
            if cm.values[i, j] >= 0.05:
                ax.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if cm.values[i, j] > 0.5 else "black")
    save_plot(fig, f"confusion_{d}")

# 3. change per class compared to the reference, in points
for d in STUDY_DATASETS:
    t = per_class[per_class.dataset == d].pivot(index="setting", columns="class", values="accuracy")
    t = (t - t.loc["reference"]).drop("reference") * 100
    t = t.reindex([n for n in STUDY_SETTINGS if n in t.index])[STUDY_DATA[d]["class_names"]]
    lim = max(1, np.nanmax(np.abs(t.values)))
    fig, ax = plt.subplots(figsize=(0.45 * t.shape[1] + 3, 0.35 * t.shape[0] + 1.5))
    im = ax.imshow(t.values, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
    ax.set(xticks=range(t.shape[1]), yticks=range(t.shape[0]), title=f"{d}: accuracy change per class vs. reference (points)")
    ax.set_xticklabels(t.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(t.index, fontsize=8)
    fig.colorbar(im, ax=ax)
    save_plot(fig, f"change_per_class_{d}")

# best and worst classes of the reference
reference_per_class = per_class[per_class.setting == "reference"].sort_values(["dataset", "accuracy"], ascending=[True, False])
reference_per_class[["dataset", "class", "accuracy"]].round(3)

---
# NEW Study: save everything for our discussion

`study/summary.md` has all tables, `study.zip` has everything (tables as CSV, plots as PNG and SVG, all adapters).

In [ ]:
# NEW Study: save tables, summary and zip
!pip install -q tabulate
tables = {"data": data_table, "classes": class_table, "accuracy_per_class": per_class, "baseline": baseline.reset_index(names="dataset"), "accuracy": accuracy_table,
          "change_vs_reference": change_table, "spread_per_group": spread}
if "xai_table" in globals():
    tables.update({"xai_per_layer": xai_table, "words": word_table, "accuracy_vs_xai": side_by_side})

summary = [f"# LoRA study\n\nreference: r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}, lr={LEARNING_RATE}, "
           f"max length {MAX_LENGTH}, seeds {STUDY_SEEDS}, quick run: {STUDY_QUICK}"]
for name, t in tables.items():
    t.to_csv(os.path.join(STUDY_DIR, f"{name}.csv"), index=False)
    if name not in ("xai_per_layer", "accuracy_per_class"):   # long tables only as CSV
        summary.append(f"## {name}\n\n" + t.round(4).to_markdown(index=False))
summary.append("## accuracy_per_class, reference\n\n" + reference_per_class[["dataset", "class", "accuracy"]].round(3).to_markdown(index=False))
summary.append("## Plots\n\n" + "\n".join(f"- {p}" for p in sorted(os.listdir(STUDY_DIR)) if p.endswith(".png")))
with open(os.path.join(STUDY_DIR, "summary.md"), "w") as f:
    f.write("\n\n".join(summary))
shutil.make_archive(STUDY_DIR, "zip", STUDY_DIR)
print("saved", STUDY_DIR + ".zip")